In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

base_tf = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.4802, 0.4481, 0.3975], [0.2302, 0.2265, 0.2262])
])

transform_aug = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.4802, 0.4481, 0.3975], [0.2302, 0.2265, 0.2262])
])

TIN_PATH = './data/tiny-imagenet-10'

def get_data(use_aug=False):
    tf = transform_aug if use_aug else base_tf
    if os.path.exists(TIN_PATH):
        full = datasets.ImageFolder(os.path.join(TIN_PATH, 'train'), transform=tf)
        test = datasets.ImageFolder(os.path.join(TIN_PATH, 'val'),   transform=base_tf)
    else:
        print("tiny imagenet not found, using cifar-10")
        full = datasets.CIFAR10('./data', train=True,  download=True, transform=tf)
        test = datasets.CIFAR10('./data', train=False, download=True, transform=base_tf)
    n_val   = int(0.2 * len(full))
    n_train = len(full) - n_val
    train, val = random_split(full, [n_train, n_val],
                              generator=torch.Generator().manual_seed(42))
    return train, val, test


train_ds, val_ds, test_ds = get_data(use_aug=False)
train_aug_ds, _, _        = get_data(use_aug=True)

train_loader     = DataLoader(train_ds,     batch_size=64, shuffle=True)
train_aug_loader = DataLoader(train_aug_ds, batch_size=64, shuffle=True)
val_loader       = DataLoader(val_ds,       batch_size=128)
test_loader      = DataLoader(test_ds,      batch_size=128)

n_classes = len(test_ds.classes) if hasattr(test_ds, 'classes') else 10
print(f"train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}, classes: {n_classes}")

# Q4.1 - Deep CNN with toggleable BatchNorm (8 layers)
class DeepCNN(nn.Module):
    def __init__(self, use_batchnorm=True, n_classes=10):
        super(DeepCNN, self).__init__()
        self.use_batchnorm = use_batchnorm

        self.conv1 = nn.Conv2d(3,   64,  3, padding=1)
        self.conv2 = nn.Conv2d(64,  64,  3, padding=1)
        self.conv3 = nn.Conv2d(64,  128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, 3, padding=1)
        self.conv5 = nn.Conv2d(128, 256, 3, padding=1)
        self.conv6 = nn.Conv2d(256, 256, 3, padding=1)
        self.conv7 = nn.Conv2d(256, 512, 3, padding=1)
        self.conv8 = nn.Conv2d(512, 512, 3, padding=1)

        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(128)
        self.bn5 = nn.BatchNorm2d(256)
        self.bn6 = nn.BatchNorm2d(256)
        self.bn7 = nn.BatchNorm2d(512)
        self.bn8 = nn.BatchNorm2d(512)

        self.pool = nn.MaxPool2d(2, 2)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(512 * 4 * 4, n_classes)

    def _apply_bn(self, x, bn):
        return bn(x) if self.use_batchnorm else x

    def forward(self, x):
        x = F.relu(self._apply_bn(self.conv1(x), self.bn1))
        x = F.relu(self._apply_bn(self.conv2(x), self.bn2))
        x = self.pool(x)
        x = F.relu(self._apply_bn(self.conv3(x), self.bn3))
        x = F.relu(self._apply_bn(self.conv4(x), self.bn4))
        x = self.pool(x)
        x = F.relu(self._apply_bn(self.conv5(x), self.bn5))
        x = F.relu(self._apply_bn(self.conv6(x), self.bn6))
        x = self.pool(x)
        x = F.relu(self._apply_bn(self.conv7(x), self.bn7))
        x = F.relu(self._apply_bn(self.conv8(x), self.bn8))
        x = self.pool(x)
        x = self.drop(x.view(x.size(0), -1))
        return self.fc(x)


m_nobn = DeepCNN(use_batchnorm=False, n_classes=n_classes).to(device)
m_bn   = DeepCNN(use_batchnorm=True,  n_classes=n_classes).to(device)
print(f"total params: {sum(p.numel() for p in m_bn.parameters()):,}")

# hook to track conv5 mean/variance across 500 batches
class ActStats:
    def __init__(self, layer):
        self.means = []
        self.vars  = []
        self._hook = layer.register_forward_hook(self._record)

    def _record(self, mod, inp, out):
        a = out.detach().cpu()
        self.means.append(a.mean().item())
        self.vars.append(a.var().item())

    def remove(self):
        self._hook.remove()


def train_with_stats(model, loader, watch_layer, epochs=10, lr=3e-4, label='', max_batches=500):
    opt     = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)
    loss_fn = nn.CrossEntropyLoss()
    stats   = ActStats(watch_layer)
    hist    = {'train_acc': [], 'val_acc': [], 'loss': []}
    bc      = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, correct, n = 0, 0, 0
        for x, y in loader:
            if bc >= max_batches:
                stats.remove()
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out  = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            n          += x.size(0)
            bc         += 1
        hist['train_acc'].append(correct / n * 100)
        hist['loss'].append(total_loss / n)
        model.eval()
        vc, vn = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                vc += (model(x).argmax(1) == y).sum().item()
                vn += x.size(0)
        hist['val_acc'].append(vc / vn * 100)
        sched.step()
        print(f"[{label}] ep {ep} | loss {total_loss/n:.4f} | "
              f"train {hist['train_acc'][-1]:.1f}% | val {hist['val_acc'][-1]:.1f}%")

    return hist, stats


EPOCHS = 15

print("=== Experiment A: NO BatchNorm ===")
hist_nobn, stats_nobn = train_with_stats(m_nobn, train_loader, watch_layer=m_nobn.conv5, epochs=EPOCHS, label='no-bn')

print("\n=== Experiment B: WITH BatchNorm ===")
hist_bn, stats_bn = train_with_stats(m_bn, train_loader, watch_layer=m_bn.conv5,epochs=EPOCHS, label='with-bn')

# Q4.1 plots
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Q4.1 - BatchNorm vs No BatchNorm', fontsize=13)

axes[0,0].plot(hist_nobn['loss'], 'r-o', ms=4, label='no BN')
axes[0,0].plot(hist_bn['loss'],   'g-s', ms=4, label='with BN')
axes[0,0].set_title('Training Loss'); axes[0,0].set_xlabel('epoch')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(hist_nobn['val_acc'], 'r-o', ms=4, label='no BN')
axes[0,1].plot(hist_bn['val_acc'],   'g-s', ms=4, label='with BN')
axes[0,1].set_title('Validation Accuracy'); axes[0,1].set_xlabel('epoch')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

n = min(500, len(stats_nobn.means), len(stats_bn.means))
axes[1,0].plot(stats_nobn.means[:n], 'r', alpha=0.7, label='no BN')
axes[1,0].plot(stats_bn.means[:n],   'g', alpha=0.7, label='with BN')
axes[1,0].set_title('conv5 Activation Mean (500 batches)')
axes[1,0].set_xlabel('batch'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(stats_nobn.vars[:n], 'r', alpha=0.7, label='no BN')
axes[1,1].plot(stats_bn.vars[:n],   'g', alpha=0.7, label='with BN')
axes[1,1].set_title('conv5 Activation Variance (500 batches)')
axes[1,1].set_xlabel('batch'); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Q4.2 - train with augmentation
def simple_train(model, loader, epochs, lr=3e-4, label=''):
    opt     = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)
    loss_fn = nn.CrossEntropyLoss()
    hist    = {'train_acc': [], 'val_acc': []}
    for ep in range(1, epochs + 1):
        model.train()
        correct, n = 0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss_fn(out, y).backward()
            opt.step()
            correct += (out.argmax(1) == y).sum().item()
            n       += x.size(0)
        hist['train_acc'].append(correct / n * 100)
        model.eval()
        vc, vn = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                vc += (model(x).argmax(1) == y).sum().item()
                vn += x.size(0)
        hist['val_acc'].append(vc / vn * 100)
        sched.step()
        print(f"[{label}] ep {ep} | train {hist['train_acc'][-1]:.1f}% | val {hist['val_acc'][-1]:.1f}%")
    return hist


model_aug = DeepCNN(use_batchnorm=True, n_classes=n_classes).to(device)
print("training with augmentation...")
hist_aug = simple_train(model_aug, train_aug_loader, epochs=EPOCHS, label='aug')

def get_test_acc(model, loader):
    model.eval()
    c, n = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            c += (model(x).argmax(1) == y).sum().item()
            n += x.size(0)
    return c / n * 100


no_aug_test = get_test_acc(m_bn,      test_loader)
aug_test    = get_test_acc(model_aug, test_loader)

print(f"\nTest Accuracy (20% split):")
print(f"  no augmentation:   {no_aug_test:.2f}%")
print(f"  with augmentation: {aug_test:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Q4.2 - With vs Without Augmentation')
eps = range(1, EPOCHS + 1)

axes[0].plot(eps, hist_bn['train_acc'],  label='no aug', marker='o', ms=3)
axes[0].plot(eps, hist_aug['train_acc'], label='aug',    marker='s', ms=3)
axes[0].set_title('Train Accuracy'); axes[0].set_xlabel('epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(eps, hist_bn['val_acc'],  label='no aug', marker='o', ms=3)
axes[1].plot(eps, hist_aug['val_acc'], label='aug',    marker='s', ms=3)
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

bars = axes[2].bar(['no aug', 'aug'], [no_aug_test, aug_test],
                   color=['steelblue', 'tomato'], alpha=0.8, width=0.4)
for bar, val in zip(bars, [no_aug_test, aug_test]):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[2].set_title('Test Accuracy (20% split)')
axes[2].set_ylabel('accuracy (%)'); axes[2].set_ylim(0, 100)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()



device: cpu
tiny imagenet not found, using cifar-10


100%|██████████| 170M/170M [00:02<00:00, 65.3MB/s]


tiny imagenet not found, using cifar-10
train: 40000, val: 10000, test: 10000, classes: 10
total params: 4,771,146
=== Experiment A: NO BatchNorm ===
